# Lab 4 — The same agent in LangGraph (EP12)

In Lab 3 we wrote every memory tier ourselves — in this lab we build the same Mem0-style agent with **LangGraph**

| Lab 3 (by hand) | LangGraph |
|---|---|
| the session's `history` (working memory) | a **checkpointer** + `thread_id` |
| the warm tier + `page_in(query)` | a **Store** + `store.search(namespace, query=…)` |
| `extract()` on every message | the `remember` node |
| the cold tier (the whole conversation) | the checkpointer keeps every message of every thread |
| — | a **namespace** per user: `("memories", user_id)` |

Every answer from the model comes from a real run.

> Source: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §17.5, §17.10.3 — CC BY-SA 4.0 · LangGraph (MIT)

> ✏️ **Starter notebook — you write the key parts**
>
> Most of the code is ready. Write only the **4 places marked `✏️ TODO`** (replace `___` or `...` with your code) — each one gives the time in the video where that code is explained (▶ m:ss). Watch the video first, then try it yourself without opening the solution.
>
> Run every cell from top to bottom. A `NameError` or `TypeError` in a cell you have not finished yet is normal.
>
> Stuck? The solution is **`ep12-lab4-langgraph-memory.en.ipynb`** (the same code as in the video).

## 0. Setup

| You need | Details |
|---|---|
| Python 3.10 or later | or Google Colab, which already has it |
| An OpenRouter API key | sign up at openrouter.ai → Keys → Create Key, then add credit (this lab costs less than 0.1 US cents) |
| Packages | `langgraph==1.2.16` (the same version as EP11), `openai`, `python-dotenv`, `numpy` |

**Adding the API key**
- **Colab:** the 🔑 Secrets icon → name `OPENROUTER_API_KEY` → turn on *Notebook access*
- **Your own machine:** create a `.env` file next to the notebook with the line `OPENROUTER_API_KEY=sk-or-...`

⚠️ Never put the key in the code, and never share your `.env` file

In [ ]:
%pip install -q "langgraph==1.2.16" "openai>=1.50" python-dotenv numpy

## 1. Models and helpers from Lab 3

Gemma 4 26B (Makora) · Qwen3 Embedding 8B (Nebius) — why these models: see Lab 1
`extract()` is Lab 3's Mem0-style fact extraction, letter for letter

In [ ]:
import os, json, operator, uuid
from typing import Annotated, TypedDict
from openai import OpenAI

def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

MODEL = "google/gemma-4-26b-a4b-it"
EXTRA = {"reasoning": {"enabled": False},
         "provider": {"order": ["makora"], "allow_fallbacks": False}}
EMBED_MODEL = "qwen/qwen3-embedding-8b"
EMBED_EXTRA = {"provider": {"order": ["nebius"], "allow_fallbacks": False}}

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key(), max_retries=5)
COST = {"chat": 0.0, "embed": 0.0}

def chat(messages):
    r = client.chat.completions.create(model=MODEL, temperature=0, extra_body=EXTRA,
                                       messages=messages)
    COST["chat"] += r.usage.cost
    return r.choices[0].message.content

def ask_json(prompt, schema):
    r = client.chat.completions.create(
        model=MODEL, temperature=0, extra_body=EXTRA,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_schema",
                         "json_schema": {"name": "reply", "strict": True, "schema": schema}})
    COST["chat"] += r.usage.cost
    return json.loads(r.choices[0].message.content)

FACTS = {"type": "object", "additionalProperties": False, "required": ["facts"],
         "properties": {"facts": {"type": "array", "items": {"type": "string"}}}}

def extract(message):
    return ask_json("You manage the long-term memory of a research assistant. From this user message, "
                    "extract the facts about the user's work, deadlines and preferences worth remembering "
                    "in future sessions, as short sentences that start with 'The user'. "
                    "Return an empty list for small talk.\nMessage: " + message, FACTS)["facts"]

In [ ]:
def embed_texts(texts):
    r = client.embeddings.create(model=EMBED_MODEL, input=list(texts), extra_body=EMBED_EXTRA)
    COST["embed"] += r.usage.cost
    return [d.embedding for d in r.data]

## 2. The Store: memory across threads

`InMemoryStore(index=…)` — keeps a `value` (dict) under a **namespace** and a **key** · `index` embeds the `text` field so it can be searched by meaning
(real systems use `PostgresStore` instead — the same API)

In [ ]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore(index={"embed": embed_texts, "dims": 4096, "fields": ["text"]})

## 3. A two-node graph: `remember` → `respond`

A LangGraph node can ask for `config` (with `thread_id`, `user_id`) and `store` — the framework passes them in

In [ ]:
class State(TypedDict):
    messages: Annotated[list[dict], operator.add]

# ✏️ TODO 1 (▶ video 1:55): put the fact into the Store under the user's namespace, with a unique key and the value {'text': fact}

def remember(state, config, *, store):
    namespace = ("memories", config["configurable"]["user_id"])
    for fact in extract(state["messages"][-1]["content"]):
        ___
        print("   📝", fact)
    return {}

SYSTEM = """You are a research assistant with long-term memory across sessions.
What you remember about this user:
{memories}
Follow the user's preferences in your memory. If your memory has nothing about something, say you don't know."""

# ✏️ TODO 2 (▶ video 2:19): search the Store under the user's namespace with the latest message, top 4

def respond(state, config, *, store):
    namespace = ("memories", config["configurable"]["user_id"])
    hits = ___
    memories = "\n".join(f"- {h.value['text']}" for h in hits) or "(nothing)"
    answer = chat([{"role": "system", "content": SYSTEM.format(memories=memories)}] + state["messages"][-8:])
    return {"messages": [{"role": "assistant", "content": answer}]}

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

graph = StateGraph(State)
graph.add_node("remember", remember)
graph.add_node("respond", respond)
graph.add_edge(START, "remember")
graph.add_edge("remember", "respond")
graph.add_edge("respond", END)

# ✏️ TODO 3 (▶ video 2:43): compile with a checkpointer (InMemorySaver) and the store

app = ___

In [ ]:
def say(text, thread, user="alice"):
    config = {"configurable": {"thread_id": thread, "user_id": user}}
    out = app.invoke({"messages": [{"role": "user", "content": text}]}, config)
    return out["messages"][-1]["content"]

## 4. Monday — thread `"monday"`

In [ ]:
monday = ["Hi! I'm writing a report on transformers for time series.",
          "Please keep your answers short.",
          "My report deadline is next Wednesday.",
          "I already read the PatchTST paper, so don't suggest it again.",
          "Thanks, that's all for today!"]
for text in monday:
    print("USER:", text)
    print("AGENT:", say(text, "monday")[:120].replace("\n", " "), "\n")

In [ ]:
print(len(app.get_state({"configurable": {"thread_id": "monday"}}).values["messages"]), "messages in 'monday'")
print(app.get_state({"configurable": {"thread_id": "tuesday"}}).values)

## 5. Tuesday — a new thread, the same memory

In [ ]:
for q in ["Any papers I should read next for my report?", "When is my report due?", "What's my dog's name?"]:
    answer = say(q, "tuesday")
    print(f"USER: {q}\nAGENT: {answer[:200]}\n   ({len(answer.split())} words)\n")

## 6. What the framework adds: a namespace per user (§17.5 Privacy)

The book: multi-user systems must **enforce access control** — one user's memories must never reach another · and users must be able to **inspect and delete** their memories

In [ ]:
print(say("When is my report due?", "bob-1", user="bob"))

In [ ]:
items = store.search(("memories", "alice"), limit=10)
for it in items:
    print(it.key[:8], it.value["text"])

# ✏️ TODO 4 (▶ video 4:51): delete the deadline memory from the Store by namespace and key

deadline = next(it for it in items if "deadline" in it.value["text"])
___
print("tuesday:  ", say("When is my report due?", "tuesday"))
print("new thread:", say("When is my report due?", "wednesday"))

In [ ]:
print({k: f"${v:.5f}" for k, v in COST.items()})

## Summary

| Lab 3 (by hand) | LangGraph | What we saw |
|---|---|---|
| `history` | checkpointer + `thread_id` | "monday" 10 messages · "tuesday" empty |
| the warm tier + `page_in` | Store + `store.search` | Tuesday answered all three |
| `extract()` | the `remember` node | 4 facts |
| — | a namespace per user | Bob: "I don't know" — Alice's memories don't leak |
| — | `store.search(ns)` / `store.delete` | inspect and delete — but the checkpointer's history must be deleted separately |

**Exercises**
1. Give the `respond` node Lab 3's `memory_search` tool, adding a `tools` node and a conditional edge
2. Use `get_state_history` on the "monday" thread (EP05 Lab 5) — how many checkpoints did the checkpointer keep?
3. Delete a whole thread with `app.checkpointer.delete_thread("tuesday")` and ask about the deadline again

**Next — EP12 Recap:** agent memory in one video